# 手撕 mHC


残差链接为 X'=X+F(X), X 为恒等变换分支，F(X) 为**残差分支**。

X 为恒等变换分支, 这个分支如果有变化 T(X), 本 notebook 称为**变换分支**，而原始的残差连接中的定义为 恒等变换 T(X) = I(X)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Residual Connection

In [2]:
def ResidualConnection(h, fun):
    h_ = h + fun(h)
    return h_

mlp = nn.Linear(256, 256)
h = torch.randn(1,256)
h_ = ResidualConnection(h, mlp)

## Multi Residual Connection

考虑有`n`条变换分支，以及加入可学习的缩放因子

$ X' = \sum_i (\beta_i h + F(h)$)

In [3]:
class MultiResidualConnection(nn.Module):
    def __init__(self, dim, n):
        super(MultiResidualConnection, self).__init__()
        self.n = n
        self.beta = nn.Parameter( torch.ones( n )* (1.0/n) ) 

    def forward(self, h, fun):
        f = fun(h)
        h_ = torch.zeros_like(f)
        for i in range(self.n):
            h_ += self.beta[i] * h + f
        return h_

MRC = MultiResidualConnection(256, 4)
h_ = MRC(h, mlp)

到这里，就达到了两个目的

1. 多变换分支+单残差分支
2. 变换分支可学习

后续的其他方法都是在扩增“学习参数”和计算方式

## Hype Connections

[HYPER-CONNECTIONS](https://arxiv.org/pdf/2409.19606)

代码来源于论文的 Appedix.I, 本代码是通过论文代码, 反向理解公式的。

In [4]:
# config 

dim = 512
rate = 2
layer_id = 10
dynamic = True

In [5]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-12):
        super(LayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        # layernorm作用在(-1) 最后一维进行归一化
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, unbiased=False, keepdim=True)
        out_mean_var = (x - mean) / torch.sqrt(var + self.eps)
        out = self.gamma * out_mean_var + self.beta # feature level
        return out
attn = nn.Linear(dim, dim)

HC 公式为

\begin{align}
\overline{\mathbf{H}} &= \texttt{norm}(\mathbf{H}) \\
\mathcal{B}(\mathbf{H})     &= 
s_\beta \circ \texttt{tanh} (\overline{\mathbf{H}} \mathbf{W}_{\beta})^\intercal + \mathbf{B} \in \mathbb{R}^{1\times n}\\
\mathcal{A}_{m}(\mathbf{H}) &= 
s_\alpha \circ \texttt{tanh} (\overline{\mathbf{H}} \mathbf{W}_{m}) + \mathbf{A}_m \in \mathbb{R}^{n\times 1} \\
\mathcal{A}_{r}(\mathbf{H})     &= 
s_\alpha \circ \texttt{tanh} (\overline{\mathbf{H}} \mathbf{W}_{r}) + \mathbf{A}_r \in \mathbb{R}^{n\times n} 
\end{align}

以上公式计算各分支的缩放因子, 这些因子用于缩放特征向量。

这些因子是参数可学习的。

In [6]:
class HyperConnection(nn.Module):
    """
    h: hyper hidden matrix (BxLxNxD)
        B: batch_size
        L: Seq_len
        N: expansion rate
        D: feature dim
    """
    def __init__(self, dim, rate, layer_id, dynamic, device=None):
        super(HyperConnection, self).__init__()

        self.rate = rate
        self.layer_id = layer_id
        self.dynamic = dynamic

        self.static_beta = nn.Parameter(torch.ones((rate,), ))

        init_alpha0 = torch.zeros((rate, 1), )
        init_alpha0[layer_id % rate, 0] = 1.
        self.static_alpha = nn.Parameter(torch.cat([init_alpha0, torch.eye((rate), )], dim=1))
       
        if self.dynamic:
            self.dynamic_alpha_fn = nn.Parameter(torch.zeros((dim, rate+1), ))# 0 投影
            self.dynamic_alpha_scale = nn.Parameter(torch.ones(1, ) * 0.01) # 1 缩放
            self.dynamic_beta_fn = nn.Parameter(torch.zeros((dim, ), ))
            self.dynamic_beta_scale = nn.Parameter(torch.ones(1, ) * 0.01)
            
            self.layer_norm = LayerNorm(dim)
            # self.layer_norm = nn.Linear(dim, dim)
    
    def width_connection(self, h):
        # get alpha and beta
        if self.dynamic:
            norm_h = self.layer_norm(h)

        # Note: 求 Am 和 Ar
        if self.dynamic:
            wc_weight = norm_h @ self.dynamic_alpha_fn
            wc_weight = F.tanh(wc_weight)
            dynamic_alpha = wc_weight  * self.dynamic_alpha_scale
            alpha = dynamic_alpha + self.static_alpha[None, None, ...]
        else:
            alpha = self.static_alpha[None, None, ...]

        # Note: 求 B
        if self.dynamic:
            dc_weight = norm_h @ self.dynamic_beta_fn
            dc_weight = F.tanh(dc_weight)
            dynamic_beta = dc_weight * self.dynamic_beta_scale
            beta = dynamic_beta + self.static_beta[None, None, ...]
        else:
            beta = self.static_beta[None, None, ...]

        # Note: 缩放因子对输入进行缩放, 处理变换分支, 变换分支有多条
        # width connection
        mix_h = alpha.transpose(-1, -2)  @  h

        return mix_h, beta
    
    def depth_connection(self, mix_h, h_o, beta):
        # Note: beta 缩放因子处理残差分支, 再与变换分支 `mix_h` 进行相加
        h = torch.einsum("blh,bln->blnh", h_o, beta) + mix_h[..., 1:, :]

        return h

In [7]:
con = HyperConnection(dim = dim, rate = rate, layer_id = layer_id, dynamic = dynamic)
print(con)

HyperConnection(
  (layer_norm): LayerNorm()
)


### forward

In [8]:
bsz = 1
seq_len = 16
h = torch.randn(bsz, seq_len, rate, dim)
mix_h, beta = con.width_connection(h)
print('h', h.shape)
print('beta', beta.shape)
print('mix_h', mix_h.shape)
f_h = attn(mix_h[...,0,:]) 
print('mix_h', f_h.shape)
out = con.depth_connection(mix_h, f_h, beta)
print('out', out.shape)

h torch.Size([1, 16, 2, 512])
beta torch.Size([1, 16, 2])
mix_h torch.Size([1, 16, 3, 512])
mix_h torch.Size([1, 16, 512])
out torch.Size([1, 16, 2, 512])


### alpha

In [9]:
norm_h = con.layer_norm(h)

if con.dynamic:

    # rate 变成 rate+1 , 实现对维度的压缩
    wc_weight = norm_h @ con.dynamic_alpha_fn  # BLNN'  <- BLND @ DN', N' = N+1
    print('wc_weight\t', wc_weight.shape) 
    wc_weight = F.tanh(wc_weight)

    # 可
    dynamic_alpha = wc_weight  * con.dynamic_alpha_scale # BLNN'
    alpha = dynamic_alpha + con.static_alpha[None, None, ...] # BLNN'
    print('alpha:\t', alpha.shape) # BLNN'

print('dynamic_alpha_fn\t', con.dynamic_alpha_fn.shape)
print('dynamic_alpha_scale\t', con.dynamic_alpha_scale.shape)
print('static_alpha\t', con.static_alpha.shape)

# alpha 链路可以看成是对每个 token feature, 压缩成一个变换因子, 对于扩展率为 n, 则有 n 个变换因子
#  BLNN', N'=N+1, 其中 1 个作为残差分支的因子

wc_weight	 torch.Size([1, 16, 2, 3])
alpha:	 torch.Size([1, 16, 2, 3])
dynamic_alpha_fn	 torch.Size([512, 3])
dynamic_alpha_scale	 torch.Size([1])
static_alpha	 torch.Size([2, 3])


In [10]:
print(con.dynamic_alpha_fn)
print(con.dynamic_alpha_scale) # 这个缩放因子来源？
print(con.static_alpha) # 这个分布目的

Parameter containing:
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        ...,
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]], requires_grad=True)
Parameter containing:
tensor([0.0100], requires_grad=True)
Parameter containing:
tensor([[1., 1., 0.],
        [0., 0., 1.]], requires_grad=True)


### beta

In [11]:
print(con.dynamic_beta_fn.shape) 
# print(con.dynamic_beta_fn)
print(con.dynamic_beta_scale)
print(con.static_beta)

torch.Size([512])
Parameter containing:
tensor([0.0100], requires_grad=True)
Parameter containing:
tensor([1., 1.], requires_grad=True)


In [12]:
if con.dynamic:
    dc_weight = norm_h @ con.dynamic_beta_fn
    print(norm_h.shape)
    print(con.dynamic_beta_fn.shape) 
    print(dc_weight.shape)
    dc_weight = F.tanh(dc_weight)

    
    dynamic_beta = dc_weight * con.dynamic_beta_scale
    beta = dynamic_beta + con.static_beta[None, None, ...]
    print(beta.shape)

torch.Size([1, 16, 2, 512])
torch.Size([512])
torch.Size([1, 16, 2])
torch.Size([1, 16, 2])


### 对比 alpha beta

遵从投影, tanh激活, 缩放, 增加static_beta

In [13]:
print(alpha.shape)
print(beta.shape)

torch.Size([1, 16, 2, 3])
torch.Size([1, 16, 2])


In [14]:
print(alpha[0,0,:,:])
print(beta[0,0,:])

tensor([[1., 1., 0.],
        [0., 0., 1.]], grad_fn=<SliceBackward0>)
tensor([1., 1.], grad_fn=<SliceBackward0>)


In [15]:
print(alpha.transpose(-1, -2).shape)
print(alpha.transpose(-1, -2)[0,0])
print(h.shape)
mix_h = alpha.transpose(-1, -2)  @  h

print(mix_h.shape)

torch.Size([1, 16, 3, 2])
tensor([[1., 0.],
        [1., 0.],
        [0., 1.]], grad_fn=<SelectBackward0>)
torch.Size([1, 16, 2, 512])
torch.Size([1, 16, 3, 512])


### depth connection

In [16]:
f_h = attn(mix_h[...,0,:]) 
print(f_h.shape)
print(beta.shape)

nh = torch.einsum("blh,bln->blnh", f_h, beta) # beta 缩放
print(nh.shape)
print(nh[0,0,0,:4])
print(nh[0,0,1,:4])
out = nh + mix_h[..., 1:, :]

print(mix_h[..., 1:, :].shape)
print(out.shape)

torch.Size([1, 16, 512])
torch.Size([1, 16, 2])
torch.Size([1, 16, 2, 512])
tensor([ 0.6105,  0.6799,  0.8503, -0.3269], grad_fn=<SliceBackward0>)
tensor([ 0.6105,  0.6799,  0.8503, -0.3269], grad_fn=<SliceBackward0>)
torch.Size([1, 16, 2, 512])
torch.Size([1, 16, 2, 512])


In [17]:
a = torch.arange(2).unsqueeze(dim=0)
b = torch.arange(4).unsqueeze(dim=0)
torch.einsum("bh,bn->bnh", a, b) 

tensor([[[0, 0],
         [0, 1],
         [0, 2],
         [0, 3]]])

### 公式对应代码表述

\begin{align}
\overline{\mathbf{H}} &= \texttt{norm}(\mathbf{H}) \\
\mathcal{B}(\mathbf{H})     &= 
s_\beta \circ \texttt{tanh} (\overline{\mathbf{H}} \mathbf{W}_{\beta})^\intercal + \mathbf{B} \in \mathbb{R}^{1\times n}\\
\mathcal{A}_{m}(\mathbf{H}) &= 
s_\alpha \circ \texttt{tanh} (\overline{\mathbf{H}} \mathbf{W}_{m}) + \mathbf{A}_m \in \mathbb{R}^{n\times 1} \\
\mathcal{A}_{r}(\mathbf{H})     &= 
s_\alpha \circ \texttt{tanh} (\overline{\mathbf{H}} \mathbf{W}_{r}) + \mathbf{A}_r \in \mathbb{R}^{n\times n} 
\end{align}

回到 HC 公式：

- B(H) 为 `beta`
- Am(H) 为 `mix_h[:,:,0,:]`
- Ar(H) 为 `mix_h[:,:,1:,:]` , 在 mHC 论文中被描述为是 $H^\text{res}$

关于 HC 矩阵, 论文有描述, 但是代码可以不构造这个“缩放因子”矩阵。

\begin{equation}
\mathcal{HC}(\mathbf{H}) = \begin{pmatrix}
\mathbf{0}_{1\times 1} & \mathcal{B}(\mathbf{H}) \\
\mathcal{A}_m(\mathbf{H}) & \mathcal{A}_r(\mathbf{H})
\end{pmatrix}
\end{equation}

In [18]:
print(alpha.shape)
print(beta.shape)
print(beta.unsqueeze(dim=2).shape)

mat_hc = torch.zeros(rate+1, rate+1 )
mat_hc[0, 1:] = beta[0,0,:] # B
mat_hc[1:, 0] = alpha[0,0,:,0] # Am
mat_hc[1:, 1:] = alpha[0,0,:,1:] # Ar
print(mat_hc)

torch.Size([1, 16, 2, 3])
torch.Size([1, 16, 2])
torch.Size([1, 16, 1, 2])
tensor([[0., 1., 1.],
        [1., 1., 0.],
        [0., 0., 1.]], grad_fn=<CopySlices>)


这里的 HC 矩阵输出与论文3.2公式(17)中的描述是对应的

\begin{equation}
\mathcal{HC}=\begin{pmatrix}
0 & 1 & 1\\
1 & 1 & 0\\
0 & 0 & 1
\end{pmatrix}.
\end{equation}

### block 里的 forward

增加一些注释补充，贴近公式

```python
# h: hyper hidden matrix (BxLxNxD)
# atten_hyper_connection, ffn_hyper_connection:  hyper-connection modules
# attn_norm, ffn_norm: normalization modules

# Attention Block

# Note: mix_h 为 (Am,Ar) 对 h 进行缩放
# Note: beta 为缩放因子
mix_h, beta = atten_hyper_connection.width_connection(h) # 内部也做一次 ln
h = attn_norm(mix_h[...,0,:])
h = self_attention(h)

# Note: 对残差分支做 dropout(h), beta 缩放
# Note: mix_h 为变换分支与残差分支进行相加
h = atten_hyper_connection.depth_connection(mix_h, dropout(h), beta)

# FFN Block
mix_h, beta = ffn_hyper_connection.width_connection(h)
h = ffn_norm(mix_h[...,0,:])
h = ffn(h)
h = ffn_hyper_connection.depth_connection(mix_h, dropout(h), beta)

```

### 总体实现逻辑

论文 Figure.8 增加了描述, 可以视为 

1. 在第0层解码块, 对特征`[bsz, seq_len, dim]`进行 repeat 了 expansion rate 次 `[bsz, seq_len, rate, dim]`
2. 每个解码块的输入输出为 `[bsz, seq_len, rate, dim]`
3. 最后一个解码块输出为 `[bsz, seq_len, rate, dim]` 在 `rate` 维度进行 `sum` 操作

### 分析与思考

1. 过去的残差连接模块是无学习参数的模块，HC 则加入下投影参数进行投影, 这个投影刚好可以增加 `n` 条变换分支
2. 每个变换分支都有独立的缩放因子，而每个特征向量都有独立的 HC 矩阵。
3. HC 增加一种新的 scale 的形式, 工程上需要分析此模块带来的 activation 显存量, 能否做到 forward/backward 可控
4. HC 增加了传统 short-cut 的复杂度，需要进一步分析数据流的方差。

对于多分枝的网络结构，是否可以以 “特征加权组合” 来进行思考？以下为经典的加权组合结构

1. Attention
2. MoE

## Manifold-Constrained Hyper-Connections

[mHC: Manifold-Constrained Hyper-Connections](https://arxiv.org/pdf/2512.24880)

1. 对 HC 矩阵进行“归一化”, 使得行列和均为1, 比如最简单的理解是：完整数独里行列都有 1,2,3,...,9
2. mHC 提到 Sinkhorn-Knopp 即为“归一化”的方法, 做法非常简单。我们会独立进行 coding。
3. mHC 作完上述操作, 在数学层面认为投影到流形空间, 增加了某种约束？
4. 工程上，可以增加重计算减少激活值, 需要增加 dualpipe 对 mHC 的微处理

总的来说 DeepSeek 肯定是测试了 HC 的方法, 发现有稳定性/工程性的问题，从而提出 mHC

方法:

\begin{equation}
    \begin{cases}
        \vec{\mathbf{x}}'_l = \text{RMSNorm}(\vec{\mathbf{x}}_l) \\
        \tilde{\mathcal{H}}_l^\text{pre} = \alpha_l^\mathrm{pre} \cdot (\vec{\mathbf{x}}'_l\phi^\mathrm{pre}_l) + \mathbf{b}_l^\mathrm{pre} \\
        \tilde{\mathcal{H}}_l^\text{post} = \alpha_l^\mathrm{post} \cdot (\vec{\mathbf{x}}'_l\phi^\mathrm{post}_l) + \mathbf{b}_l^\mathrm{post} \\
        \tilde{\mathcal{H}}_l^\text{res} = \alpha_l^\mathrm{res} \cdot \text{mat}(\vec{\mathbf{x}}'_l\phi^\mathrm{res}_l) + \mathbf{b}_l^\mathrm{res}, \\
    \end{cases}
\end{equation}

其中 mat 是 reshape 方法

\begin{equation}
    \begin{cases}
        \mathcal{H}_l^\text{pre} = \sigma(\tilde{\mathcal{H}}_l^\text{pre}) \\
        \mathcal{H}_l^\text{post} = 2\sigma(\tilde{\mathcal{H}}_l^\text{post}) \\
        \mathcal{H}_l^\text{res} = \text{Sinkhorn-Knopp}(\tilde{\mathcal{H}}_l^\text{res}),
    \end{cases}
\end{equation}

其中\text{Sinkhorn-Knopp}进行归一化

对应到 HC 论文的公式描述:

1. $\mathcal{H}_l^\text{pre}$: 为 alpha，Am
2. $\mathcal{H}_l^\text{post}$: 为 beta
3. $\mathcal{H}_l^\text{res}$: 为 Ar

### 实现

In [19]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-12):
        super(RMSNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        mean = (x**2).mean(-1, keepdim=True)
        out_mean = x / torch.sqrt(mean + self.eps) # root mean square
        out = self.gamma * out_mean 
        return out

In [20]:
class ManifoldHyperConnection(nn.Module):
    """
    此代码 follow HC 写法, 可以大致理解写法就行
    后续会按照 mHC 论文写出 fuse 版本的实现

    h: hyper hidden matrix (BxLxNxD)
        B: batch_size
        L: Seq_len
        N: expansion rate
        D: feature dim
    """
    def __init__(self, dim, rate, layer_id):
        super(ManifoldHyperConnection, self).__init__()

        self.rate = rate
        self.layer_id = layer_id
        self.dynamic = dynamic
        self.dim = dim

        self.nc = self.rate * self.dim
        self.n2 = self.rate * self.rate

        # 将输入 flat 化
        self.norm = RMSNorm(dim*rate)

        self.beta_pre = nn.Parameter(torch.ones(self.rate))
        self.beta_post = nn.Parameter(torch.ones(self.rate))
        self.beta_res = nn.Parameter(torch.ones(self.rate, self.rate))
       
        self.alpha_pre_scale = nn.Parameter(torch.ones(1) * 0.01) # 1 缩放
        self.alpha_post_scale = nn.Parameter(torch.ones(1) * 0.01)
        self.alpha_res_scale = nn.Parameter(torch.ones(1) * 0.01)

        self.w_pre = nn.Parameter(torch.zeros( self.nc, self.rate ) )
        self.w_post = nn.Parameter(torch.zeros( self.nc, self.rate ) )
        self.w_res = nn.Parameter(torch.zeros( self.nc, self.n2 ) )
        
    def width_connection(self, h, res_norm):
        """
        res_norm 为 H_res 的归一化方法
        """
        # get alpha and beta
        B, L, N, D = h.shape
        h = h.reshape(B, L, N*D)
        norm_h = self.norm(h)

        # Note: 求 pre
        pre_weight = norm_h @ self.w_pre
        dynamic_pre = pre_weight  * self.alpha_pre_scale
        H_pre = dynamic_pre + self.beta_pre[None, None, ...]
       
        # Note: 求 post
        post_weight = norm_h @ self.w_post
        dynamic_beta = post_weight * self.alpha_post_scale
        H_post = dynamic_beta + self.beta_post[None, None, ...]

        # Note: 求 res
        res_weight = norm_h @ self.w_res
        # mat() operation
        res_weight = res_weight.reshape(B, L, self.rate, self.rate)
        dynamic_res = res_weight * self.alpha_res_scale 
        H_res = dynamic_res + self.beta_res[None, None, ...]

        # constrained
        H_pre = F.sigmoid(H_pre)
        H_post = 2 * F.sigmoid(H_post)
        H_res = res_norm(H_res)

        # Note: 缩放因子对输入进行缩放, 处理变换分支, 变换分支有多条
        # width connection
        h_pre = H_pre.transpose(-1, -2)  @  h
        h_res = H_res.transpose(-1, -2)  @  h.reshape(B, L, N, D)

        return h_pre, h_res, H_post
    
    # def depth_connection(self, mix_h, h_o, beta):
    #     # Note: beta 缩放因子处理残差分支, 再与变换分支 `mix_h` 进行相加
    #     h = torch.einsum("blh,bln->blnh", h_o, beta) + mix_h[..., 1:, :]

    #     return h

In [21]:
mHC = ManifoldHyperConnection(dim=dim, rate=rate, layer_id=layer_id)

def res_norm_basic(x):
    return x/torch.max(torch.abs(x))

h_pre, h_res, H_post = mHC.width_connection(h, res_norm_basic)
print(h_pre.shape)
print(h_res.shape)
print(H_post.shape)

torch.Size([1, 2, 1024])
torch.Size([1, 16, 2, 512])
torch.Size([1, 16, 2])


## sinkhorn_knopp 归一化

原论文

To this end, we restrict $\mathcal{H}^\text{res}_{l}$ to be a doubly stochastic matrix, which has non-negative entries where both the rows and columns sum to 1. Formally, let $\mathcal{M}^\mathrm{res}$ denote the manifold of doubly stochastic matrices (also known as the Birkhoff polytope).
We constrain $\mathcal{H}^\text{res}_{l}$ to $\mathcal{P}_{\mathcal{M}^\mathrm{res}}(\mathcal{H}^\text{res}_{l})$, defined as:
$$
\begin{equation}
    \mathcal{P}_{\mathcal{M}^\mathrm{res}}(\mathcal{H}^\text{res}_{l}) \coloneq \left\{ \mathcal{H}^\text{res}_{l} \in \mathbb{R}^{n \times n} \mid \mathcal{H}^\text{res}_{l}\mathbf{1}_n = \mathbf{1}_n, \ \mathbf{1}^\top_n\mathcal{H}^\text{res}_{l} = \mathbf{1}^\top_n, \ \mathcal{H}^\text{res}_{l} \geq 0 \right\},
\end{equation}
$$
where $\mathbf{1}_n$ represents the $n$-dimensional vector of all ones.




---

Sinkhorn-Knopp算法是一种通过**交替行列归一化**将非负矩阵转换为双随机矩阵（doubly stochastic matrix）的迭代方法。它是矩阵平衡（matrix balancing）问题的经典解法。

---

### 1. **核心目标**
给定一个非负矩阵 $ A \in \mathbb{R}^{n \times n}_+ $，找到两个对角矩阵 $ D_1 $ 和 $ D_2 $，使得：
$$
P = D_1 A D_2
$$
满足：
- $ P $ 的**每一行和** = 1（行随机）
- $ P $ 的**每一列和** = 1（列随机）
即 $ P $ 是一个**双随机矩阵**。

---

#### 2. **算法步骤**
假设 $ A $ 的元素均非负，且是完全正的（即支持双随机化）。

**初始化**：  
设 $ A^{(0)} = A $，迭代 $ k = 0, 1, 2, \dots $

**交替归一化**：
1. **行归一化**：  
   $$
   r_i^{(k)} = \sum_j A_{ij}^{(k)}, \quad A_{ij}^{\text{row-scaled}} = \frac{A_{ij}^{(k)}}{r_i^{(k)}}
   $$
   使得每行和为 1。

2. **列归一化**：  
   $$
   c_j^{(k)} = \sum_i A_{ij}^{\text{row-scaled}}, \quad A_{ij}^{(k+1)} = \frac{A_{ij}^{\text{row-scaled}}}{c_j^{(k)}}
   $$
   使得每列和为 1。

重复以上两步直到收敛（行和与列和都接近 1）。

---

#### 3. **数学表达**
设对角矩阵 $ D_1 = \text{diag}(u) $，$ D_2 = \text{diag}(v) $，目标为：
$$
P = \text{diag}(u) \, A \, \text{diag}(v)
$$
满足：
$$
P \mathbf{1} = \mathbf{1}, \quad P^T \mathbf{1} = \mathbf{1}
$$
其中 $\mathbf{1}$ 是全1向量。

这等价于：
$$
u \odot (A v) = \mathbf{1}, \quad v \odot (A^T u) = \mathbf{1}
$$
其中 $\odot$ 是逐元素乘法。

**迭代形式**（Sinkhorn迭代）：
$$
u^{(t+1)} = \frac{1}{A v^{(t)}}, \quad v^{(t+1)} = \frac{1}{A^T u^{(t+1)}}
$$
这里的除法是逐元素除法。


### SK迭代实现

In [22]:
def sinkhorn_knopp_basic(A, it=100, eps=1e-8):
    n, _ = A.shape
    u = torch.ones(n)
    v = torch.ones(n)
    
    # 确保 A 非负
    A = torch.clamp(A, min=eps)
    
    for _ in range(it):
        v_temp = v.clone()
        u = 1.0 / (A @ v_temp + eps)
        v = 1.0 / (A.t() @ u + eps)
    
    P = torch.diag(u) @ A @ torch.diag(v)
    return P, u, v

A = torch.randn(3,3)
A_, _, _ = sinkhorn_knopp_basic(A, it=10)
print(A_)
print('it=10\t', A_.sum(dim=0), A_.sum(dim=1))
A_, _, _ = sinkhorn_knopp_basic(A, it=1000)
print('it=1000\t', A_.sum(dim=0), A_.sum(dim=1))

tensor([[3.4346e-01, 6.9291e-07, 6.0065e-01],
        [5.4765e-01, 1.1049e-06, 3.9935e-01],
        [7.3578e-02, 1.0000e+00, 2.1633e-09]])
it=10	 tensor([0.9647, 1.0000, 1.0000]) tensor([0.9441, 0.9470, 1.0736])
it=1000	 tensor([0.9984, 1.0000, 1.0000]) tensor([0.9992, 0.9992, 1.0000])


### SK-批量版本迭代实现

采用 exp 来做确保 A 非负, 大约迭代20次就收敛

In [23]:
def sinkhorn_knopp_batched(A, it=1000, eps=1e-8):
    """
    批量版本的Sinkhorn-Knopp算法
    """
    
    batch_size, n, _, = A.shape
    
    # 初始化
    u = torch.ones(batch_size, n)
    v = torch.ones(batch_size, n)
    
    # 确保 A 非负
    # A = torch.clamp(A, min=eps)
    A = torch.exp(A)
    
    for _ in range(it):
        # 更新u
        v_temp = v.unsqueeze(2)  # (B, n, 1)
        Av = torch.bmm(A, v_temp).squeeze(2)  # (B, n)
        u = 1.0 / (Av + eps)
        
        # 更新v
        u_temp = u.unsqueeze(2)  # (B, n, 1)
        At_u = torch.bmm(A.transpose(1, 2), u_temp).squeeze(2)
        v = 1.0 / (At_u + eps)
    
    # 构建双随机矩阵
    U = torch.diag_embed(u)  # (B, n, n)
    V = torch.diag_embed(v)  # (B, n, n)
    P = torch.bmm(torch.bmm(U, A), V)
    
    if batch_size == 1:
        P = P.squeeze(0)
        u = u.squeeze(0)
        v = v.squeeze(0)
    
    return P, u, v

A = torch.randn(2,3,3)

# example1
A_, _, _ = sinkhorn_knopp_batched(A, it=2)
print(A_.shape)
print('it=10\t', A_[0].sum(dim=0), A_[0].sum(dim=1))

# example2
A_, _, _ = sinkhorn_knopp_batched(A, it=100)
print('it=1000\t', A_[0].sum(dim=0), A_[0].sum(dim=1))

torch.Size([2, 3, 3])
it=10	 tensor([1.0000, 1.0000, 1.0000]) tensor([0.9819, 1.0180, 1.0000])
it=1000	 tensor([1., 1., 1.]) tensor([1., 1., 1.])


## mHC Fuse 实现

在之前实现的版本里分别处理, pre, post, pre, 

mHC 可以对相同计算进行合并

$$
\begin{align}
    \phi_l                                                                      &: \text{tfloat32}          &&[nC, n^2+2n]                                                              \label{eq:fuse:1}\\
    \vec{\mathbf{x}}_l                                                                      &: \text{bfloat16}          &&[1, nC]                                                                      \label{eq:fuse:2}\\
    \alpha_l^\mathrm{pre}, \alpha_l^\mathrm{post}, \alpha_l^\mathrm{res}                                                    &: \text{float32}           &&\text{Scalars}                                                          \label{eq:fuse:3}\\
    \mathbf{b}_l                                                                      &: \text{float32}           &&[1, n^2+2n]                                                                  \label{eq:fuse:4}\\
    \left[{\tilde{\tilde{\mathcal{H}}}^{\mathrm{pre}}_{l}}, {\tilde{\tilde{\mathcal{H}}}^{\mathrm{post}}_{l}}, {\tilde{\tilde{\mathcal{H}}}^{\mathrm{res}}_{l}}\right]   &: \text{float32}           &&= \vec{\mathbf{x}}_l\phi_l                                                  \label{eq:fuse:5}\\
    r                                                                               &: \text{float32}           &&= \left\|\vec{\mathbf{x}}_l\right\|_2 / \sqrt{nC}                                                       \label{eq:fuse:6}\\
    \left[\tilde{\mathcal{H}}^{\mathrm{pre}}_{l}, \tilde{\mathcal{H}}^{\mathrm{post}}_{l}, \tilde{\mathcal{H}}^{\mathrm{res}}_{l}\right]         &: \text{float32}           &&= 1/r \left[\alpha_l^\mathrm{pre}{\tilde{\tilde{\mathcal{H}}}^{\mathrm{pre}}_{l}}, \alpha_l^\mathrm{post}{\tilde{\tilde{\mathcal{H}}}^{\mathrm{post}}_{l}}, \alpha_l^\mathrm{res}{\tilde{\tilde{\mathcal{H}}}^{\mathrm{res}}_{l}}\right] + \mathbf{b}_l \label{eq:fuse:7}\\
    \mathcal{H}^{\mathrm{pre}}_{l}                                                                      &: \text{float32}           &&= \sigma\left(\tilde{\mathcal{H}}^{\mathrm{pre}}_{l}\right)                                   \label{eq:fuse:8}\\
    \mathcal{H}^{\mathrm{post}}_{l}                                                                      &: \text{float32}           &&= 2\sigma\left(\tilde{\mathcal{H}}^{\mathrm{post}}_{l}\right)                                  \label{eq:fuse:9}\\
    \mathcal{H}^{\mathrm{res}}_{l}                                                                      &: \text{float32}           &&= \text{Sinkhorn-Knopp}\left(\tilde{\mathcal{H}}^{\mathrm{res}}_{l}\right)   \label{eq:fuse:10}
\end{align}
$$

In [24]:
import math
class ManifoldHyperConnectionFuse(nn.Module):
    """
    h: hyper hidden matrix (BxLxNxD)
        B: batch_size
        L: Seq_len
        N: expansion rate
        D: feature dim
    """
    def __init__(self, dim, rate, layer_id, max_sk_it):
        super(ManifoldHyperConnectionFuse, self).__init__()

        self.n = rate
        self.dim = dim

        self.nc = self.n * self.dim
        self.n2 = self.n * self.n

        # 将输入 flat 化
        self.norm = RMSNorm(dim*rate)

        # 参数
        self.w = nn.Parameter(torch.zeros(self.nc, self.n2 + 2*self.n))
        self.alpha = nn.Parameter(torch.ones(3) * 0.01)
        self.beta = nn.Parameter(torch.zeros(self.n2 + 2*self.n) * 0.01)

        # 最大sk迭代
        self.max_sk_it = max_sk_it

    def mapping(self, h, res_norm):
        B, L, N, D = h.shape

        # 1.vectorize
        h_vec = h.reshape(B, L, N*D)

        # 2.projection
        H = h_vec @ self.w

        # 3. scaled
        r = h_vec.norm() / math.sqrt(self.nc)
        r_ = 1 / r

        # 4. mapping
        n = N
        H_pre = r_ * H[:,:, :n] * self.alpha[0] + self.beta[:n]
        H_post = r_ * H[:,:, n:2*n] * self.alpha[1] + self.beta[n:2*n]
        H_res = r_ * H[:,:, 2*n:] * self.alpha[2] + self.beta[2*n:]

        # 5. final constrained mapping 
        H_pre = F.sigmoid(H_pre)
        H_post = 2 * F.sigmoid(H_post)
        
        H_res = H_res.reshape(B, L, N, N)
        H_res, _, _ = res_norm(H_res.reshape(B*L, N, N), self.max_sk_it)
        H_res = H_res.reshape(B, L, N, N)

        return H_pre, H_post, H_res

    def process(self, h, H_pre, H_res):
        h_pre = H_pre.unsqueeze(dim=2) @ h
        h_res = H_res.transpose(2,3) @ h
        return h_pre, h_res

    def depth_connection(self, H_pre, h_out, beta):
        post_mapping = beta.unsqueeze(dim=-1) @ h_out
        out = post_mapping + h_res
        return out
        

max_sk_it = 20
h = torch.randn(bsz, seq_len, rate, dim)
mHC = ManifoldHyperConnectionFuse(dim = dim, 
                                  rate = rate, 
                                  layer_id = layer_id,
                                  max_sk_it = max_sk_it)
H_pre, H_post, H_res = mHC.mapping(h, sinkhorn_knopp_batched)
print(H_pre.shape)
print(H_post.shape)
print(H_res.shape)

h_pre, h_res = mHC.process(h, H_pre, H_res)
print(h_pre.shape)
print(h_res.shape)

torch.Size([1, 16, 2])
torch.Size([1, 16, 2])
torch.Size([1, 16, 2, 2])
torch.Size([1, 16, 1, 512])
torch.Size([1, 16, 2, 512])


### forward

In [25]:
H_pre, H_post, H_res = mHC.mapping(h, sinkhorn_knopp_batched)
h_pre, h_res = mHC.process(h, H_pre, H_res)
h_out = attn(h_pre) 
out = mHC.depth_connection(h_res, h_out, beta=H_post)
print('out', out.shape)

# 其他 block 同理

out torch.Size([1, 16, 2, 512])


### 分析和思考

1. sinkhorn_knopp 迭代看成是归一化手段， 在 Muon 优化中用 `msign` 算子进行约束，`msign` 的计算通过`Newton-schulz`迭代求的
3. HC、mHC 可以看成是残差分支   $ \beta F( \alpha h )$ 与多个 $ \beta_i h $ 进行加权组合, $\beta_i$ 为第 $i$ 各分支的缩放
4. 完整的计算过程残差分支保留激活者, 其他变换分支都用重计算技巧减少显存, 其他分支都是比较小的计算过程

疑问：

1. 初始化、归一化如何设置，保证输入输出的 forward、backward 稳定？
2. mHC 解决 HC 的稳定性问题，HC的不稳定来源哪里？

## 总结

1. 新链接范式为残差分支 + 多变换分支
2. 残差链接变为可参数学习学习，学习的难点是平衡各个变换分支的权重和为 1
3. 需要了解 HC 就能快速写出 mHC